# Evaluating Retrieval Quality

RAG fails when **retrieval** fails — the LLM cannot cite facts that never arrived in context. Before tuning prompts or swapping models, measure whether the **right chunks** appear in top-k results.

**Retrieval evaluation** uses a labeled query set: for each question, you know which chunk ids should be retrieved. Metrics like **Recall@k**, **MRR**, and **Precision@k** quantify search quality independently of generation.

This notebook covers building gold-standard eval sets, metric definitions and implementations, miss analysis, comparing k and chunk settings, and separating retrieval eval from end-to-end RAG eval.

Prerequisites: **01–08** in this section. Use results from notebook **08** before migrating vector DB vendors. Full RAG eval continues in **08. RAG Fundamentals**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import chromadb
from openai import OpenAI

openai_client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

---

## 1. Retrieval Eval vs Generation Eval

```
User query
    │
    ▼
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│  Retrieval  │ ──► │  Prompt +   │ ──► │  Generated  │
│  (top-k)    │     │  LLM        │     │  answer     │
└─────────────┘     └─────────────┘     └─────────────┘
      ▲                                         ▲
 Recall@k, MRR                          Faithfulness,
 (this notebook)                        citation accuracy
```

| Layer | Question | Typical metrics |
|-------|----------|-----------------|
| **Retrieval** | Did the right chunks appear in top-k? | Recall@k, MRR, nDCG |
| **Generation** | Did the LLM answer correctly using context? | Exact match, LLM-as-judge |

High retrieval recall with bad answers → fix **prompting** (**04. Prompt Engineering**). Low recall with good-looking answers → fix **chunking, embeddings, or hybrid search**.

---

## 2. Building a Labeled Eval Set

### 2.1 Gold-Standard Format

Each row is one test query with known relevant chunk ids:

| Field | Example |
|-------|---------|
| `query` | "How do I run migrations?" |
| `relevant_ids` | `["c2", "c5"]` |
| `query_type` | `factual`, `keyword`, `paraphrase` |
| `notes` | "Alembic upgrade head" |

### 2.2 Where Labels Come From

| Source | Strength |
|--------|----------|
| **Subject experts** | High trust; slow to produce |
| **Support tickets** | Real user language |
| **Manual review** | Run search, mark correct chunks |
| **Synthetic from docs** | Fast bootstrap; may miss phrasing diversity |

### 2.3 Coverage Checklist

Include queries that stress different failure modes:

| Query type | Example | Tests |
|------------|---------|-------|
| **Direct** | "JWT authentication" | Basic semantic match |
| **Paraphrase** | "How do clients prove identity?" | Embedding generalization |
| **Keyword-heavy** | "ECONNREFUSED pytest" | Hybrid need (notebook 08) |
| **Multi-relevant** | "database schema changes" | Multiple valid chunks |
| **Negative / edge** | "billing refunds" | Should miss (no chunk) |

Aim for **30–100** labeled queries before major architecture decisions; **10** is enough for learning demos.

---

## 3. Metrics

### 3.1 Recall@k (Hit Rate)

Fraction of queries where **at least one** relevant id appears in the top-k retrieved ids:

$$\text{Recall@k} = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{1}\left[ \text{relevant}(q) \cap \text{top-}k(q) \neq \emptyset \right]$$

For RAG prototypes, **Recall@3** and **Recall@5** are the most actionable — they mirror how many chunks you inject into the prompt.

### 3.2 Precision@k

Among the k retrieved ids, what fraction are relevant (averaged over queries):

$$\text{Precision@k} = \frac{1}{|Q|} \sum_{q} \frac{|\text{relevant}(q) \cap \text{top-}k(q)|}{k}$$

High recall + low precision → noisy context; the LLM must filter irrelevant chunks.

### 3.3 Mean Reciprocal Rank (MRR)

Rewards ranking the **first** relevant hit higher:

$$\text{MRR} = \frac{1}{|Q|} \sum_{q} \frac{1}{\text{rank of first relevant hit}}$$

If no relevant hit, reciprocal rank is 0.

### 3.4 nDCG (Overview)

**Normalized Discounted Cumulative Gain** handles **graded** relevance (highly relevant vs somewhat relevant). Use when labels have scores, not just binary ids.

| Metric | When to use |
|--------|-------------|
| **Recall@k** | Default RAG metric |
| **Precision@k** | Context window is tight |
| **MRR** | Rank of first hit matters |
| **nDCG** | Graded relevance labels |

---

## 4. Corpus and Eval Set

Sample chunks aligned with this curriculum — FastAPI Notes API, Alembic, JWT, pytest. Includes a **confuser** chunk (OAuth) to stress auth queries.

In [ ]:
CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI and stores notes in memory for demos.", "source": "api-docs", "doc_type": "api"},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling new revisions.", "source": "db-docs", "doc_type": "api"},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Clients send Authorization: Bearer <token>.", "source": "security-docs", "doc_type": "policy"},
    {"id": "c4", "text": "Pytest fixtures share database setup across tests. Use conftest.py for session-scoped engines.", "source": "test-docs", "doc_type": "runbook"},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live database schema and drafts revision files.", "source": "db-docs", "doc_type": "api"},
    {"id": "c6", "text": "OAuth2 authorization code flow redirects users to an identity provider; distinct from simple JWT bearer usage.", "source": "security-docs", "doc_type": "policy"},
    {"id": "c7", "text": "SQLAlchemy models in the Notes API define Note rows with id, title, and body columns.", "source": "api-docs", "doc_type": "api"},
    {"id": "c8", "text": "Alembic revision files live in the versions/ directory; each file has a revision id and down_revision chain.", "source": "db-docs", "doc_type": "api"},
]

EVAL_SET = [
    {"query": "How do I run database migrations?", "relevant_ids": ["c2", "c5", "c8"], "query_type": "direct"},
    {"query": "What framework is the Notes API built with?", "relevant_ids": ["c1"], "query_type": "direct"},
    {"query": "How does API authentication work?", "relevant_ids": ["c3"], "query_type": "paraphrase"},
    {"query": "How do I share test database setup across pytest files?", "relevant_ids": ["c4"], "query_type": "direct"},
    {"query": "Alembic autogenerate workflow", "relevant_ids": ["c5"], "query_type": "keyword"},
    {"query": "Where are migration revision files stored?", "relevant_ids": ["c8"], "query_type": "paraphrase"},
    {"query": "SQLAlchemy Note model columns", "relevant_ids": ["c7"], "query_type": "keyword"},
    {"query": "Bearer token in Authorization header", "relevant_ids": ["c3"], "query_type": "keyword"},
]

print(f"Corpus: {len(CHUNKS)} chunks | Eval set: {len(EVAL_SET)} queries")

---

## 5. Index Corpus in Chroma

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    r = openai_client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [d.embedding for d in r.data]


client = chromadb.Client()
try:
    client.delete_collection("retrieval_eval")
except Exception:
    pass

col = client.create_collection("retrieval_eval")
texts = [c["text"] for c in CHUNKS]
metadatas = [{"source": c["source"], "doc_type": c["doc_type"]} for c in CHUNKS]
col.add(
    ids=[c["id"] for c in CHUNKS],
    documents=texts,
    embeddings=embed(texts),
    metadatas=metadatas,
)
print("Indexed", col.count(), "chunks")

---

## 6. Metric Implementations

In [ ]:
def retrieve_ids(query: str, k: int = 5, where: dict | None = None) -> list[str]:
    kwargs = {"query_embeddings": embed([query]), "n_results": k, "include": ["distances"]}
    if where:
        kwargs["where"] = where
    r = col.query(**kwargs)
    return r["ids"][0]


def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> bool:
    return bool(relevant & set(retrieved[:k]))


def precision_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    top = retrieved[:k]
    if not top:
        return 0.0
    return len(relevant & set(top)) / len(top)


def reciprocal_rank(retrieved: list[str], relevant: set[str]) -> float:
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


def evaluate(eval_set: list[dict], k: int = 5, where: dict | None = None) -> dict:
    details = []
    recalls, precisions, rrs = [], [], []
    for row in eval_set:
        got = retrieve_ids(row["query"], k=k, where=where)
        relevant = set(row["relevant_ids"])
        hit = recall_at_k(got, relevant, k)
        prec = precision_at_k(got, relevant, k)
        rr = reciprocal_rank(got, relevant)
        recalls.append(hit)
        precisions.append(prec)
        rrs.append(rr)
        details.append({
            "query": row["query"],
            "query_type": row.get("query_type", "unknown"),
            "got": got,
            "relevant": sorted(relevant),
            "hit": hit,
            "precision": prec,
            "rr": rr,
        })
    n = len(eval_set)
    return {
        f"recall@{k}": sum(recalls) / n,
        f"precision@{k}": sum(precisions) / n,
        "mrr": sum(rrs) / n,
        "details": details,
    }

---

## 7. Run Baseline Evaluation

In [ ]:
K = 5
results = evaluate(EVAL_SET, k=K)

print(f"Recall@{K}:    {results[f'recall@{K}']:.0%}")
print(f"Precision@{K}: {results[f'precision@{K}']:.0%}")
print(f"MRR:           {results['mrr']:.3f}")
print()
for d in results["details"]:
    mark = "OK" if d["hit"] else "MISS"
    print(f"[{mark}] ({d['query_type']}) {d['query']}")
    print(f"      retrieved: {d['got']}")
    print(f"      expected:  {d['relevant']}")

Inspect **MISS** rows first — they reveal chunk gaps, confuser chunks, or k that is too small.

---

## 8. Compare k Values

Recall usually rises with k; precision often falls. Pick k to match your RAG context budget.

In [ ]:
print(f"{'k':>3}  {'Recall':>8}  {'Precision':>10}  {'MRR':>8}")
print("-" * 36)
for k in [1, 3, 5, 8]:
    r = evaluate(EVAL_SET, k=k)
    print(f"{k:3d}  {r[f'recall@{k}']:8.0%}  {r[f'precision@{k}']:10.0%}  {r['mrr']:8.3f}")

If Recall@1 is low but Recall@5 is high, the right chunk exists but ranks below position 1 — try better chunk text, hybrid search, or re-ranking.

---

## 9. Break Down by Query Type

In [ ]:
from collections import defaultdict

by_type: dict[str, list[bool]] = defaultdict(list)
for d in results["details"]:
    by_type[d["query_type"]].append(d["hit"])

print(f"Recall@{K} by query_type:")
for qtype, hits in sorted(by_type.items()):
    rate = sum(hits) / len(hits)
    print(f"  {qtype:12s}  {rate:.0%}  ({sum(hits)}/{len(hits)})")

Keyword-heavy queries often underperform pure semantic search — a signal to add **hybrid BM25** (notebook 08).

---

## 10. A/B Comparison: Metadata Filter

Simulate restricting search to `doc_type=api` — helps migration queries but may hurt security queries if labels expect `policy` chunks.

In [ ]:
baseline = evaluate(EVAL_SET, k=K)
filtered = evaluate(EVAL_SET, k=K, where={"doc_type": "api"})

print("Comparison: no filter vs doc_type=api")
print(f"  Recall@{K}  baseline={baseline[f'recall@{K}']:.0%}  filtered={filtered[f'recall@{K}']:.0%}")
print(f"  MRR         baseline={baseline['mrr']:.3f}  filtered={filtered['mrr']:.3f}")


Filters can **raise** precision on domain-specific evals or **destroy** recall if relevant chunks sit outside the filter — always eval with filters you deploy in production.

---

## 11. Simulating a Chunking Regression

Add a vague **mega-chunk** that mixes topics — recall on focused queries often drops (diluted embedding).

In [ ]:
BAD_CHUNK = {
    "id": "bad1",
    "text": (
        "This document covers FastAPI, Alembic migrations, JWT tokens, pytest fixtures, "
        "OAuth flows, SQLAlchemy models, and deployment notes all in one paragraph."
    ),
    "source": "misc",
    "doc_type": "api",
}

col.add(
    ids=[BAD_CHUNK["id"]],
    documents=[BAD_CHUNK["text"]],
    embeddings=embed([BAD_CHUNK["text"]]),
    metadatas=[{"source": BAD_CHUNK["source"], "doc_type": BAD_CHUNK["doc_type"]}],
)

after_bad = evaluate(EVAL_SET, k=K)
print(f"Recall@{K} before bad chunk: {results[f'recall@{K}']:.0%}")
print(f"Recall@{K} after bad chunk:  {after_bad[f'recall@{K}']:.0%}")
print(f"MRR before: {results['mrr']:.3f} | after: {after_bad['mrr']:.3f}")

Use this pattern to compare **chunk strategies** (notebook 04) or **embedding models** (notebook 03) — index variant A vs B, run the same `EVAL_SET`, compare metrics.

---

## 12. Error Analysis Workflow

```
1. List all MISS queries
2. Read retrieved vs expected chunk text side-by-side
3. Classify failure:
     • chunk missing answer text  → re-chunk / enrich source
     • confuser chunk wins        → split topics, add metadata filter
     • right chunk rank 4+        → increase k, hybrid, re-rank
     • query wording OOD          → add paraphrases to eval + training data
4. Fix one variable; re-run eval
5. Track metrics in a spreadsheet or experiment log
```

| Failure class | Fix |
|---------------|-----|
| Missing terms in chunk | Smaller chunks with overlap, or enrich headers |
| Semantic drift | Larger model, domain fine-tune |
| Exact token miss | Hybrid BM25 + RRF |
| Wrong tenant/version | Metadata pre-filter |
| ANN recall loss | Exact search on subset or tune index params |

---

## 13. Using Eval Results

| Outcome | Next step |
|---------|-----------|
| Miss on migration queries | Ensure chunk mentions Alembic + upgrade |
| Keyword queries fail | Hybrid search (notebook 08) |
| Low recall globally | Chunk size, overlap, embedding model |
| High recall, bad LLM answers | Generation layer — prompts, citations |
| Filtered worse than baseline | Relax or redesign metadata schema |

**Vendor choice:** Run the same eval set on Chroma vs pgvector vs managed options before migrating (notebook 08).

---

## 14. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Eval set = training queries only | Overfit to phrasing | Diverse paraphrases |
| Only end-to-end RAG score | Can't isolate retrieval | Separate Recall@k |
| Tiny eval (3 queries) | False confidence | Grow to 30+ |
| Labels on doc ids not chunk ids | Wrong granularity | Label chunk ids |
| Ignore k in metric | k=1 vs k=5 tells different stories | Report multiple k |
| No regression after ingest change | Silent quality drop | CI eval on every index build |

---

## 15. Summary

Build a **labeled query set** with diverse query types. Measure **Recall@k**, **Precision@k**, and **MRR** before blaming the LLM. Inspect misses, compare k and filters, and A/B chunk or index changes on the same eval set.

Retrieval eval is the feedback loop for chunking (04), embeddings (03), metadata (07), and vendor choice (08).

Next: **10. Production Patterns for Embeddings and Vector Stores** — versioning, batching, logging, and cost at scale.